## Latent space organization: VAE v3 vs Simple v2

This notebook extracts **global latents** (encoder mean `mu`, 128-D) from a random subset of the **test memmap** split,
then runs:
- clustering (**KMeans**, **HDBSCAN**)
- 2D dimensionality reduction (**UMAP**)
- interpretation by coloring with **attribute bins** (polyphony, rhythmic intensity, velocity, note density)

Outputs are written to `artifacts/latent_viz/`:
- `latent_space_report.json`
- `*_umap_*.png`
- `latent_space_embeddings.npz`


In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path

# Make sure local package imports work (repo layout: ml/src/musicgen)
import sys
sys.path.insert(0, str((Path.cwd().parents[1] / "ml" / "src").resolve()))

# Speed up repeated runs by using writable caches
os.environ.setdefault("NUMBA_CACHE_DIR", str((Path.cwd().parents[1] / "artifacts" / "numba_cache").resolve()))
os.environ.setdefault("MPLCONFIGDIR", str((Path.cwd().parents[1] / "artifacts" / "mpl_cache").resolve()))

from musicgen.analysis.latent_space_viz import LatentVizConfig, run_latent_space_viz


In [ ]:
# Keep this small to run fast locally.
cfg = LatentVizConfig(
    n_samples=300,
    batch_size=32,
    seed=123,
    ckpt_vae_v3="ml/src/musicgen/runs/vae_v3/ckpt.pt",
    ckpt_simple_v2="ml/src/musicgen/runs/simple_v2/ckpt.pt",
    out_dir="artifacts/latent_viz",
    embedding="mu",
)

report_path = run_latent_space_viz(cfg)
report_path


In [ ]:
report = json.loads(Path(report_path).read_text(encoding="utf-8"))
report["kmeans"]["vae_v3"]["silhouette"], report["kmeans"]["simple_v2"]["silhouette"]


In [ ]:
# Display generated images
from IPython.display import Image, display

out_dir = Path(cfg.out_dir)
for name in [
    "vae_v3_umap_kmeans.png",
    "vae_v3_umap_hdbscan.png",
    "simple_v2_umap_kmeans.png",
    "simple_v2_umap_hdbscan.png",
    "vae_v3_umap_attr_note_density.png",
    "simple_v2_umap_attr_note_density.png",
]:
    p = out_dir / name
    if p.exists():
        display(Image(filename=str(p)))
